# Create Planet.com orders for landslide after-images

**Purpose:** Searches Planet's Data API for scenes covering each landslide incident
AOI in the post-event window, then submits a clip order named `incident_<ID>_after`
for each. Uses the `analytic_sr_udm2` bundle clipped to the incident AOI.

**Run before `planet_orders_download.ipynb`:** Use the same `start_idx` / `end_idx`
range when downloading. Leave both as `0` to order **all** incidents in the CSV.

This notebook runs on **Kaggle**.

## Setup (one time)
1. Get your Planet API key from https://www.planet.com/account/#/
2. In Kaggle: **Add-ons → Secrets** → add a secret named `planet_api_key`
3. Ensure the landslide incidents CSV dataset is attached as a Kaggle input dataset.


In [ ]:
# --------------------------------------------------------------------
# Imports
# --------------------------------------------------------------------
import os, re, time
import pandas as pd
import requests
from requests.adapters import HTTPAdapter, Retry
from kaggle_secrets import UserSecretsClient


In [ ]:
# --------------------------------------------------------------------
# Configuration — edit these before running
# --------------------------------------------------------------------

# CSV containing landslide incidents
input_csv = "/kaggle/input/datasets/sanjayashrestha123/landslide-reproted/landslides_from_2018_to_2026.csv"

# Index range into the CSV (df.iloc[start_idx:end_idx]).
# Set both to 0 to order ALL incidents in the CSV.
start_idx = 0
end_idx   = 0

# --- Planet ---
ORDERS_URL  = "https://api.planet.com/compute/ops/orders/v2"
DATA_URL    = "https://api.planet.com/data/v1"
ITEM_TYPE   = "PSScene"
BUNDLE_TYPE = "analytic_sr_udm2"
MAX_SCENES_PER_ORDER = 10    # cap scenes included in one order

# Post-event search window
POST_BUFFER_DAYS = 5    # skip scenes within N days of incident
POST_DAYS        = 45   # search up to N days after incident

MAX_CLOUD_COVER = 0.5   # Planet-level cloud cover filter (0–1)

# AOI clamping (same as extracting_data.ipynb)
MAX_AOI_DEG = 0.1


In [ ]:
# --------------------------------------------------------------------
# Authenticate with Planet API
# --------------------------------------------------------------------
user_secrets = UserSecretsClient()
PLANET_API_KEY = user_secrets.get_secret("planet_api_key")

def make_planet_session():
    s = requests.Session()
    s.auth = (PLANET_API_KEY, "")
    retries = Retry(
        total=5,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["GET", "POST"]),
        respect_retry_after_header=True,
    )
    s.mount("https://", HTTPAdapter(max_retries=retries))
    return s

session = make_planet_session()
r = session.get(ORDERS_URL, timeout=60)
r.raise_for_status()
print("Planet authentication OK.")

# --------------------------------------------------------------------
# Fetch all existing order names to avoid re-submitting
# --------------------------------------------------------------------
print("Fetching existing orders...")
existing_order_names = set()
url = ORDERS_URL
page = 0
while url:
    resp = session.get(url, timeout=120)
    resp.raise_for_status()
    data = resp.json()
    for o in data.get("orders", []):
        existing_order_names.add(o.get("name", ""))
    url = data.get("_links", {}).get("_next")
    page += 1
print(f"Found {len(existing_order_names)} existing orders across {page} page(s).")


In [ ]:
# --------------------------------------------------------------------
# Load the landslide incidents CSV and apply index range
# --------------------------------------------------------------------
df = pd.read_csv(input_csv)
if start_idx == 0 and end_idx == 0:
    df_range = df.copy()
    print(f"Loaded all {len(df_range)} incidents from CSV.")
else:
    df_range = df.iloc[start_idx:end_idx].copy()
    print(f"Loaded {len(df_range)} incidents from CSV rows {start_idx}:{end_idx}.")

df_range['incident_on'] = pd.to_datetime(df_range['incident_on'], dayfirst=True)

already_ordered = [
    int(row['id'])
    for _, row in df_range.iterrows()
    if f"incident_{int(row['id'])}_after" in existing_order_names
]
to_order = [
    int(row['id'])
    for _, row in df_range.iterrows()
    if f"incident_{int(row['id'])}_after" not in existing_order_names
]
print(f"Already have orders: {len(already_ordered)}")
print(f"To order:            {len(to_order)}")


In [ ]:
# --------------------------------------------------------------------
# Helper functions
# --------------------------------------------------------------------
def clamp_aoi(min_lon, min_lat, max_lon, max_lat):
    lon_span = max_lon - min_lon
    lat_span = max_lat - min_lat
    if lon_span <= MAX_AOI_DEG and lat_span <= MAX_AOI_DEG:
        return min_lon, min_lat, max_lon, max_lat
    cx = (min_lon + max_lon) / 2
    cy = (min_lat + max_lat) / 2
    half = MAX_AOI_DEG / 2
    return cx - half, cy - half, cx + half, cy + half


def search_planet_scenes(session, aoi_geom, start_date, end_date):
    """Search Planet Data API v1 for scenes overlapping the AOI in the given window."""
    filter_payload = {
        "item_types": [ITEM_TYPE],
        "filter": {
            "type": "AndFilter",
            "config": [
                {"type": "GeometryFilter",  "field_name": "geometry", "config": aoi_geom},
                {"type": "DateRangeFilter", "field_name": "acquired",
                 "config": {"gte": start_date, "lte": end_date}},
                {"type": "RangeFilter",     "field_name": "cloud_cover",
                 "config": {"lte": MAX_CLOUD_COVER}},
            ],
        },
    }
    resp = session.post(f"{DATA_URL}/quick-search", json=filter_payload, timeout=60)
    resp.raise_for_status()
    return [f["id"] for f in resp.json().get("features", [])]


def submit_order(session, order_name, item_ids, aoi_geom):
    """Submit a Planet order with a clip tool and return the order dict."""
    payload = {
        "name": order_name,
        "products": [
            {"item_ids": item_ids, "item_type": ITEM_TYPE, "product_bundle": BUNDLE_TYPE}
        ],
        "tools": [{"type": "clip", "parameters": {"aoi": aoi_geom}}],
    }
    resp = session.post(ORDERS_URL, json=payload, timeout=60)
    resp.raise_for_status()
    return resp.json()


# --------------------------------------------------------------------
# Create orders for incidents not yet ordered
# --------------------------------------------------------------------
created = skipped = failed = 0
errors = []

for inc_id in to_order:
    row = df_range[df_range['id'] == inc_id].iloc[0]
    order_name = f"incident_{inc_id}_after"

    incident_date = row['incident_on']
    min_lon, min_lat, max_lon, max_lat = clamp_aoi(
        row['min_lon'], row['min_lat'], row['max_lon'], row['max_lat'])
    aoi_geom = {
        "type": "Polygon",
        "coordinates": [[[min_lon, min_lat], [max_lon, min_lat],
                          [max_lon, max_lat], [min_lon, max_lat],
                          [min_lon, min_lat]]],
    }
    after_start = (incident_date + pd.DateOffset(days=POST_BUFFER_DAYS)).strftime('%Y-%m-%dT00:00:00Z')
    after_end   = (incident_date + pd.DateOffset(days=POST_DAYS)).strftime('%Y-%m-%dT23:59:59Z')

    try:
        item_ids = search_planet_scenes(session, aoi_geom, after_start, after_end)
        if not item_ids:
            errors.append(f"incident_{inc_id}: no scenes found in window {after_start} – {after_end}")
            failed += 1
            continue
        capped = item_ids[:MAX_SCENES_PER_ORDER]
        submit_order(session, order_name, capped, aoi_geom)
        created += 1
        print(f"  [OK] {order_name}: {len(capped)} scene(s)")
        time.sleep(0.5)   # gentle rate limiting
    except Exception as e:
        errors.append(f"incident_{inc_id}: {e}")
        failed += 1
        print(f"  [FAIL] incident_{inc_id}: {e}")

print(f"\nDone: {created} created, {len(already_ordered)} already existed, {failed} failed.")
if errors:
    print("\nErrors:")
    for e in errors[:10]:
        print(f"  - {e}")


In [ ]:
# --------------------------------------------------------------------
# Status summary of all orders matching the incident pattern
# --------------------------------------------------------------------
from collections import Counter

all_orders = []
url = ORDERS_URL
while url:
    resp = session.get(url, timeout=120)
    resp.raise_for_status()
    data = resp.json()
    all_orders.extend(data.get("orders", []))
    url = data.get("_links", {}).get("_next")

pattern = re.compile(r'^incident_(\d+)_after$')
relevant = [o for o in all_orders if pattern.match(o.get("name", ""))]
states = Counter(o.get("state") for o in relevant)
print(f"Orders matching 'incident_*_after': {len(relevant)}")
print("States:", dict(states))

if to_order and created > 0:
    new_names = {f"incident_{i}_after" for i in to_order}
    new_orders = [o for o in relevant if o.get("name") in new_names]
    new_states = Counter(o.get("state") for o in new_orders)
    print(f"\nOf the {created} newly submitted orders:")
    print("  States:", dict(new_states))
